In [1]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(
    n_workers=4,
    threads_per_worker=1,
    memory_limit="2GB"
)
client = Client(cluster)

client


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 7.45 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:64550,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:64570,Total threads: 1
Dashboard: http://127.0.0.1:64574/status,Memory: 1.86 GiB
Nanny: tcp://127.0.0.1:64553,


In [2]:
print(client.dashboard_link)


http://127.0.0.1:8787/status


In [3]:
import numpy as np
import dask.array as da

# A large numeric array split into row chunks
x = da.from_array(np.random.default_rng(0).normal(size=(200_000, 50)),
                  chunks=(20_000, 50))

# Build a lazy computation
col_means = x.mean(axis=0)

# Trigger execution
result = col_means.compute()
result[:5]

C:\Users\corne\Job\packt_python_4e\parallel_computing\dask_env\Lib\site-packages\distributed\client.py:3375: UserWarning: Sending large graph of size 76.30 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


array([ 1.23384874e-03, -3.47863122e-04, -2.34752573e-03,  3.00144582e-03,
       -6.89085388e-05])

In [4]:
import dask.dataframe as dd
from pathlib import Path

data_path = Path("retail_event.csv")

ddf_retail = dd.read_csv(data_path, assume_missing=True)

# Make header robust for real-world CSVs
ddf_retail = ddf_retail.rename(columns=lambda c: c.strip())
ddf_retail["event_time"] = dd.to_datetime(ddf_retail["event_time"], errors="coerce")

print(ddf_retail.dtypes)
print("Partitions:", ddf_retail.npartitions)

event_time                 datetime64[ns]
user_id                           float64
session_id                        float64
country                   string[pyarrow]
city                      string[pyarrow]
device                    string[pyarrow]
channel                   string[pyarrow]
ad_campaign               string[pyarrow]
referrer                  string[pyarrow]
product_id                        float64
category                  string[pyarrow]
price                             float64
quantity                          float64
discount_pct                      float64
shipping_cost                     float64
pages_viewed                      float64
time_on_site_sec                  float64
clicked_recommendation            float64
payment_method            string[pyarrow]
final_amount                      float64
purchased                         float64
dtype: object
Partitions: 1


In [5]:
ddf_retail = ddf_retail.repartition(npartitions=8)
print("Partitions after repartition:", ddf_retail.npartitions)

Partitions after repartition: 8


In [6]:
avg_amount_by_country = ddf_retail.groupby("country")["final_amount"].mean()
avg_amount_by_country_result = avg_amount_by_country.compute()
avg_amount_by_country_result

country
AU    142.366613
ID    137.737333
IN    138.044919
JP    140.976724
KR    140.133862
MY    136.966926
PH    136.722215
SG    136.121411
TH    139.820655
VN    138.117800
Name: final_amount, dtype: float64

In [7]:
import dask.bag as db
import json

b = db.read_text("events.jsonl").map(json.loads)

countries = b.filter(lambda r: "country" in r).pluck("country")

country_counts = countries.frequencies()
country_counts_result = country_counts.compute()

country_counts_result[:10]

[('ID', 16878),
 ('JP', 4884),
 ('VN', 3870),
 ('PH', 3007),
 ('TH', 3456),
 ('IN', 3934),
 ('KR', 3415),
 ('SG', 2422),
 ('AU', 3807),
 ('MY', 2832)]

In [8]:
b.take(1)

({'event_time': '2025-12-20T22:01:21',
  'user_id': 128768,
  'session_id': 7157618,
  'event_type': 'add_to_cart',
  'country': 'ID',
  'metadata': {'device': 'desktop', 'channel': 'social', 'campaign': 'none'}},)

In [9]:
n = b.count().compute()
n

50000

In [10]:
referrers = b.map(lambda r: r.get("referrer", "unknown")).map(lambda x: x.strip().lower())
referrers.take(5)

('unknown', 'unknown', 'unknown', 'unknown', 'instagram')

In [11]:
countries = b.filter(lambda r: "country" in r).pluck("country")
countries.take(5)

('ID', 'JP', 'ID', 'JP', 'VN')

In [12]:
b.filter(lambda r: r.get("event_type") == "purchase" and r.get("country") == "ID").count().compute()

1040

In [13]:
unique_countries = (
    b.filter(lambda r: "country" in r)
     .pluck("country")
     .distinct()
     .compute()
)

In [14]:
amounts = b.filter(lambda r: "amount" in r).pluck("amount")
total_amount = amounts.sum().compute()
avg_amount = amounts.mean().compute()

In [15]:
amount_sum = (
    b.filter(lambda r: "amount" in r)
     .pluck("amount")
     .fold(lambda acc, x: acc + x, initial=0.0)
     .compute()
)
amount_sum

180671.43000000014

In [16]:
records = (
    b.map(lambda r: {
        "event_time": r.get("event_time"),
        "country": r.get("country"),
        "event_type": r.get("event_type"),
        "user_id": r.get("user_id"),
        "amount": r.get("amount")
    })
)

ddf_events = records.to_dataframe(
    meta={
        "event_time": "object",
        "country": "object",
        "event_type": "object",
        "user_id": "int64",
        "amount": "float64"
    }
)

ddf_events.head()

,event_time,country,event_type,user_id,amount
0,2025-12-20T22:01:21,ID,add_to_cart,128768,NaN
1,2025-12-12T04:33:46,JP,page_view,100848,NaN
2,2025-11-25T22:41:45,ID,page_view,184261,NaN
3,2025-10-24T03:33:42,JP,checkout,111422,11.95
4,2025-08-11T07:03:38,VN,page_view,38537,NaN


In [17]:
data_path = Path("retail_event.csv")

ddf_retail = dd.read_csv(data_path, assume_missing=True)
ddf_retail = ddf_retail.rename(columns=lambda c: c.strip())

# Parse timestamps
ddf_retail["event_time"] = dd.to_datetime(ddf_retail["event_time"], errors="coerce")

# Clean string columns
ddf_retail["referrer"] = ddf_retail["referrer"].fillna("unknown").str.strip().str.lower()
ddf_retail["channel"] = ddf_retail["channel"].fillna("unknown").str.strip().str.lower()

# Handle missing numeric values
ddf_retail["discount_pct"] = ddf_retail["discount_pct"].fillna(0.0)
ddf_retail["shipping_cost"] = ddf_retail["shipping_cost"].fillna(0.0)

# Type conversions for IDs (nullable integers)
ddf_retail["user_id"] = ddf_retail["user_id"].astype("Int64")
ddf_retail["session_id"] = ddf_retail["session_id"].astype("Int64")
ddf_retail["product_id"] = ddf_retail["product_id"].astype("Int64")

# Convert target label to small integer type
ddf_retail["purchased"] = ddf_retail["purchased"].fillna(0).astype("int8")

# Reduce to a clean modelling table
features = ddf_retail[[
    "event_time", "country", "device", "channel",
    "final_amount", "discount_pct", "shipping_cost",
    "pages_viewed", "time_on_site_sec", "clicked_recommendation",
    "purchased"
]]

features.head()

,event_time,country,device,channel,final_amount,discount_pct,shipping_cost,pages_viewed,time_on_site_sec,clicked_recommendation,purchased
0,2025-07-01 00:00:21,ID,mobile,organic,279.71,0.200,6.59,4.0,72.0,0.0,0
1,2025-07-01 00:02:56,ID,mobile,social,266.97,0.000,3.69,4.0,86.0,0.0,0
2,2025-07-01 00:04:37,IN,tablet,paid_search,64.36,0.078,1.30,4.0,113.0,1.0,1
3,2025-07-01 00:04:45,ID,desktop,display,51.82,0.000,3.05,9.0,39.0,0.0,0
4,2025-07-01 00:06:12,TH,desktop,social,48.48,0.000,1.38,7.0,51.0,0.0,0


In [18]:
# Compute an aggregate rather than the full table
avg_amount = features.groupby("country")["final_amount"].mean().compute()
avg_amount

country
AU    142.366613
ID    137.737333
IN    138.044919
JP    140.976724
KR    140.133862
MY    136.966926
PH    136.722215
SG    136.121411
TH    139.820655
VN    138.117800
Name: final_amount, dtype: float64

In [19]:
# Persist an intermediate result if it will be reused
features_persisted = features.persist()

In [20]:
# Write to disk instead of materializing a large DataFrame in memory
features.to_parquet("output/retail_events_cleaned/", write_index=False)

In [21]:
import joblib
from dask.distributed import Client, LocalCluster
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Start a local Dask client
client = Client(LocalCluster(n_workers=4, threads_per_worker=1))

# Select numeric features and compute them into pandas for scikit-learn
X = features[[
    "final_amount", "discount_pct", "shipping_cost",
    "pages_viewed", "time_on_site_sec", "clicked_recommendation"
]].fillna(0).compute()

y = features["purchased"].compute()

grid = GridSearchCV(LogisticRegression(max_iter=500), {"C": [0.1, 1, 10]}, cv=5, n_jobs=-1)

with joblib.parallel_backend("dask"):
    grid.fit(X, y)

C:\Users\corne\Job\packt_python_4e\parallel_computing\dask_env\Lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 64175 instead
  warnings.warn(


In [22]:
from dask_ml.wrappers import Incremental
from sklearn.linear_model import SGDClassifier

feature_cols = [
    "final_amount", "discount_pct", "shipping_cost",
    "pages_viewed", "time_on_site_sec", "clicked_recommendation"
]

X_da = features[feature_cols].fillna(0).to_dask_array(lengths=True)
y_da = features["purchased"].to_dask_array(lengths=True)

base = SGDClassifier(loss="log_loss", random_state=7)
model = Incremental(base)

model.fit(X_da, y_da, classes=np.array([0, 1], dtype=np.int8))


,estimator,SGDClassifier...andom_state=7)
,scoring,None
,shuffle_blocks,True
,random_state,None
,assume_equal_chunks,True
,predict_meta,None
,predict_proba_meta,None
,transform_meta,None
,"loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'log_loss'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001


In [23]:
from dask_ml.model_selection import train_test_split, GridSearchCV
from dask_ml.preprocessing import StandardScaler
from dask_ml.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score, classification_report

# Split using dask-ml
X_train, X_test, y_train, y_test = train_test_split(
    X_da, y_da, test_size=0.2, random_state=7, shuffle=True
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define model and small search space
clf = LogisticRegression(max_iter=500)

param_grid = {"C": [0.1, 1.0, 10.0]}
grid = GridSearchCV(LogisticRegression(max_iter=500), param_grid, cv=5, n_jobs=-1)

# Fit in parallel
with joblib.parallel_backend("dask"):
   grid.fit(X_train, y_train)

best_model = grid.best_estimator_

# Predictions are Dask Arrays, compute when we need final metrics
y_pred = best_model.predict(X_test).compute()
y_true = y_test.compute()

print("Best params:", grid.best_params_)
print(classification_report(y_true, y_pred))

C:\Users\corne\Job\packt_python_4e\parallel_computing\dask_env\Lib\site-packages\dask_glm\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Best params: {'C': 0.1}
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     36607
           1       0.00      0.00      0.00      3393

    accuracy                           0.92     40000
   macro avg       0.46      0.50      0.48     40000
weighted avg       0.84      0.92      0.87     40000



C:\Users\corne\Job\packt_python_4e\parallel_computing\dask_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\corne\Job\packt_python_4e\parallel_computing\dask_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\corne\Job\packt_python_4e\parallel_computing\dask_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this b

In [24]:
#Set Moding Engine

import os
os.environ["MODIN_ENGINE"] = "Dask"

In [25]:
import modin.pandas as pd
import modin.config as cfg

print("Engine:", cfg.Engine.get())
print("Default partitions:", cfg.NPartitions.get())

Engine: Dask
Default partitions: 20


In [26]:
cfg.NPartitions.put(8)

In [27]:
from pathlib import Path
import modin.pandas as pd

data_path = Path("retail_event.csv")

df = pd.read_csv(data_path)

# Filter
df_small = df[df["final_amount"] > 0]

# Groupby aggregation
avg_amount_by_country = df_small.groupby("country")["final_amount"].mean()

# Join example using a small lookup table
country_region = pd.DataFrame(
    {"country": ["ID", "SG", "MY", "TH", "VN", "PH", "JP", "KR", "IN", "AU"],
     "region":  ["SEA","SEA","SEA","SEA","SEA","SEA","EA","EA","SA","OCE"]}
)

joined = df_small.merge(country_region, on="country", how="left")

In [28]:
from pathlib import Path
import time

def benchmark(pd_module, data_path: Path):
    t0 = time.perf_counter()
    df = pd_module.read_csv(data_path)
    t1 = time.perf_counter()

    grp = df[df["final_amount"] > 0].groupby("country")["final_amount"].mean()
    _ = grp.head(5)
    t2 = time.perf_counter()

    lookup = pd_module.DataFrame(
        {"country": ["ID","SG","MY","TH","VN","PH","JP","KR","IN","AU"],
         "region":  ["SEA","SEA","SEA","SEA","SEA","SEA","EA","EA","SA","OCE"]}
    )
    merged = df.merge(lookup, on="country", how="left")
    _ = merged.head(5)
    t3 = time.perf_counter()

    return {
        "read_csv": t1 - t0,
        "groupby":  t2 - t1,
        "merge":    t3 - t2,
        "total":    t3 - t0
    }

def print_comparison(pandas_times, modin_times, decimals=2):
    rows = ["read_csv", "groupby", "merge", "total"]

    def r(x):  # round helper
        return round(float(x), decimals)

    print(f"{'Step':<12} {'pandas (s)':>12} {'modin (s)':>12} {'speedup':>10}")
    print("-" * 50)

    for step in rows:
        p = float(pandas_times[step])
        m = float(modin_times[step])
        speed = (p / m) if m > 0 else float("inf")
        print(f"{step:<12} {r(p):>12} {r(m):>12} {r(speed):>10}x")

# ----------------------------
# Run benchmark
# ----------------------------
data_path = Path("retail_event.csv")

# Warm-up runs (reduces one-time overhead noise)
import pandas as pandas_pd
_ = benchmark(pandas_pd, data_path)

import os
os.environ["MODIN_ENGINE"] = "Dask"
import modin.pandas as modin_pd
_ = benchmark(modin_pd, data_path)

# Timed runs
pandas_times = benchmark(pandas_pd, data_path)
modin_times = benchmark(modin_pd, data_path)

print_comparison(pandas_times, modin_times, decimals=2)


Step           pandas (s)    modin (s)    speedup
--------------------------------------------------
read_csv             0.99         0.56       1.77x
groupby              0.07         1.73       0.04x
merge                0.14         1.02       0.13x
total                 1.2         3.31       0.36x


In [29]:
import ray

context = ray.init()

2026-01-23 16:03:36,414	INFO worker.py:2007 -- Started a local Ray instance.


In [30]:
import time
import math

@ray.remote
def compute_metric(x: float) -> float:
    time.sleep(0.05)
    return math.log1p(x) * math.sqrt(x)

refs = [compute_metric.remote(i) for i in range(1, 51)]
values = ray.get(refs)
values[:5]

[0.6931471805599453,
 1.5536723984241867,
 2.4011322677058873,
 3.2188758248682006,
 4.006495972522874]

In [31]:
lookup = np.random.rand(1_000_000).astype("float32")
lookup_ref = ray.put(lookup)

@ray.remote
def score_batch(x: np.ndarray, lookup_obj) -> float:
    return float(x.mean() + lookup_obj[0])

batches = [np.random.rand(200_000).astype("float32") for _ in range(8)]
refs = [score_batch.remote(b, lookup_ref) for b in batches]
ray.get(refs)


[0.6811219453811646,
 0.6817379593849182,
 0.6815661191940308,
 0.6818987131118774,
 0.6820356845855713,
 0.6822206377983093,
 0.6817341446876526,
 0.6817139387130737]

In [32]:
@ray.remote
class ModelWorker:
    def __init__(self):
        self.bias = 0.1

    def predict(self, x: np.ndarray) -> np.ndarray:
        return x + self.bias

worker = ModelWorker.remote()
x = np.random.rand(5).astype("float32")

y = ray.get(worker.predict.remote(x))
y


array([0.1044701 , 0.56267995, 0.33244804, 0.38743287, 1.050174  ],
      dtype=float32)

In [33]:
@ray.remote(num_cpus=2)
def cpu_heavy(i: int) -> int:
    time.sleep(0.2)
    return i

# Reserve a GPU when we have one
@ray.remote(num_gpus=1)
def gpu_step(i: int) -> int:
    return i


In [36]:

import random

@ray.remote
def process_one(i: int) -> float:
    time.sleep(random.uniform(0.05, 0.2))
    return i * 0.25

# Launch tasks
refs = [process_one.remote(i) for i in range(10)]

# Collect results as tasks complete
results = []
while refs:
    ready, refs = ray.wait(refs, num_returns=1)
    results.append(ray.get(ready[0]))

results


[0.75, 0.0, 1.25, 0.25, 0.5, 2.25, 1.0, 1.5, 2.0, 1.75]

In [37]:
from pathlib import Path
import importlib
import ray
import numpy as np

from sklearn.linear_model import LogisticRegression

ray.shutdown()
ray.init(ignore_reinit_error=True)

data_path = Path("retail_event.csv")

feature_cols = [
    "final_amount", "discount_pct", "shipping_cost",
    "pages_viewed", "time_on_site_sec", "clicked_recommendation"
]

@ray.remote
def fit_one_country(country: str, path_str: str):
    pd = importlib.import_module("pandas")
    df = pd.read_csv(path_str)

    df = df[df["country"] == country].copy()
    df = df.dropna(subset=["purchased"])
    if len(df) < 200 or df["purchased"].nunique() < 2:
        return {"country": country, "status": "skipped", "rows": int(len(df))}

    X = df[feature_cols].fillna(0.0).to_numpy(dtype=np.float32)
    y = df["purchased"].fillna(0).astype("int8").to_numpy()

    model = LogisticRegression(max_iter=200)
    model.fit(X, y)

    acc = float(model.score(X, y))  # simple training accuracy
    return {"country": country, "status": "ok", "rows": int(len(df)), "train_acc": acc}

# Driver: list countries once
pd = importlib.import_module("pandas")
countries = sorted(pd.read_csv(data_path, usecols=["country"])["country"].dropna().unique())

results = ray.get([fit_one_country.remote(c, str(data_path)) for c in countries])

# Keep only successful results and show a compact summary
ok = [r for r in results if r["status"] == "ok"]
ok_sorted = sorted(ok, key=lambda r: r["train_acc"], reverse=True)

ok_sorted[:10]

2026-01-23 16:55:14,208	INFO worker.py:2007 -- Started a local Ray instance.


[{'country': 'SG',
  'status': 'ok',
  'rows': 10166,
  'train_acc': 0.9182569348809758},
 {'country': 'TH',
  'status': 'ok',
  'rows': 14066,
  'train_acc': 0.9180292904877009},
 {'country': 'AU',
  'status': 'ok',
  'rows': 15836,
  'train_acc': 0.9160772922455166},
 {'country': 'MY',
  'status': 'ok',
  'rows': 12028,
  'train_acc': 0.9157798470236116},
 {'country': 'PH',
  'status': 'ok',
  'rows': 11910,
  'train_acc': 0.9151973131821999},
 {'country': 'VN',
  'status': 'ok',
  'rows': 15957,
  'train_acc': 0.913266904806668},
 {'country': 'ID',
  'status': 'ok',
  'rows': 70375,
  'train_acc': 0.9127104795737122},
 {'country': 'IN',
  'status': 'ok',
  'rows': 15889,
  'train_acc': 0.9127069041475234},
 {'country': 'JP',
  'status': 'ok',
  'rows': 19958,
  'train_acc': 0.9121154424291011},
 {'country': 'KR',
  'status': 'ok',
  'rows': 13815,
  'train_acc': 0.9116178067318133}]